# 第 9 讲：分类模型与机器学习入门

> 落成分类模型示例代码：训练/测试划分、逻辑回归、决策树和混淆矩阵。

## 课件导学

**任务情境**：分类模型不只看准确率，还要看正类召回和混淆矩阵是否符合问题风险。

**关键概念**

- 分类标签
- 训练/测试划分
- 逻辑回归
- 决策树
- 混淆矩阵

**本讲产物**

- classification_metrics.csv
- confusion_matrix.csv
- confusion_matrix.png

## 资料链接与阅读抓手

下面这些链接优先选官方文档或稳定资料。课前先看标题和示例，课堂中遇到参数或概念再回查。

- [scikit-learn 监督学习指南](https://scikit-learn.org/stable/supervised_learning.html)：建立分类与回归的整体地图。
- [LogisticRegression 文档](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)：查看逻辑回归参数和适用条件。
- [DecisionTreeClassifier 文档](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)：理解树模型的深度、分裂准则和过拟合风险。

## 使用说明

- 先从上到下运行全部单元。
- 每讲都会生成一个 `outputs/lesson-xx/` 目录，用于保存图表、表格或报告片段。
- 示例数据均为教学用合成数据，真实比赛中应替换为题目附件数据。

In [ ]:
LESSON_ID = "lesson-09"

from pathlib import Path
import os
import math
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

OUTPUT_ROOT = Path(os.environ.get("COURSE_OUTPUT_DIR", REPO_ROOT / "outputs")) / LESSON_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
print(f"Output directory: {OUTPUT_ROOT}")

## 可运行示例与讲解路线

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

X_arr, y = make_classification(
    n_samples=320, n_features=5, n_informative=3, n_redundant=1,
    weights=[0.62, 0.38], random_state=42
)
X = pd.DataFrame(X_arr, columns=[f"x{i}" for i in range(1, 6)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
models = {
    "logistic": LogisticRegression(max_iter=1000),
    "decision_tree": DecisionTreeClassifier(max_depth=4, random_state=42),
}
rows = []
for name, clf in models.items():
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    rows.append({"model": name, "accuracy": accuracy_score(y_test, pred), "recall": recall_score(y_test, pred)})
metrics = pd.DataFrame(rows)
metrics.to_csv(OUTPUT_ROOT / "classification_metrics.csv", index=False)
metrics

In [ ]:
best = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
pred = best.predict(X_test)
cm = confusion_matrix(y_test, pred)
pd.DataFrame(cm, index=["actual_0", "actual_1"], columns=["pred_0", "pred_1"]).to_csv(OUTPUT_ROOT / "confusion_matrix.csv")
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title("Decision tree confusion matrix")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "confusion_matrix.png", dpi=160)
plt.show()

## 实战环节

**课堂任务**

- 比较逻辑回归和决策树的 recall。
- 解释一个假阴性或假阳性的业务后果。
- 调整决策树深度并观察混淆矩阵变化。

**课后挑战**：把评价指标从 accuracy 改为 F1 或 recall 优先，并说明原因。

**验收清单**

- 是否使用训练/测试划分
- 是否输出混淆矩阵
- 是否根据题目风险选择指标

In [ ]:
lesson_resources = [{'title': 'scikit-learn 监督学习指南', 'url': 'https://scikit-learn.org/stable/supervised_learning.html', 'reading_note': '建立分类与回归的整体地图。'}, {'title': 'LogisticRegression 文档', 'url': 'https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html', 'reading_note': '查看逻辑回归参数和适用条件。'}, {'title': 'DecisionTreeClassifier 文档', 'url': 'https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html', 'reading_note': '理解树模型的深度、分裂准则和过拟合风险。'}]
lesson_deliverables = ['classification_metrics.csv', 'confusion_matrix.csv', 'confusion_matrix.png']
lesson_checklist = ['是否使用训练/测试划分', '是否输出混淆矩阵', '是否根据题目风险选择指标']

pd.DataFrame(lesson_resources).to_csv(OUTPUT_ROOT / "lesson_resources.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"deliverable": lesson_deliverables}).to_csv(OUTPUT_ROOT / "lesson_deliverables.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"check_item": lesson_checklist}).to_csv(OUTPUT_ROOT / "lesson_checklist.csv", index=False, encoding="utf-8-sig")
print("Saved courseware resource map, deliverables, and checklist.")